# Scheme Saathi: government scheme summariser

> Turn a dense scheme notice into a scheme card in the reader's own language: read aloud, shareable on WhatsApp, and answerable by voice.

**Bodhan models:** indic-ocr, indic-translate, indic-transliterate, indic-speak, indic-transcribe
**Languages shown:** English notice; Hindi, Tamil, Bengali and Marathi cards; a Hindi voice question
**Format:** notebook (this walkthrough) + Streamlit app (`app.py`) + CLI (`scheme_summariser.py`)

```
notice.pdf ──indic-ocr──▶ layout blocks ──headings──▶ scheme card (verbatim lines)
                                                            │
                           indic-translate, one call/section│
             ┌──────────────────────────────────────────────┤
             ▼                                              ▼
   card in Hindi ──indic-speak──▶ audio per section    roman-script copy (script="roman") for WhatsApp

 spoken question ──indic-transcribe──┐
 typed "kaun se documents chahiye" ──indic-transliterate──┴──▶ indic-translate (→ en) ──▶ match card lines ──▶ translate + speak
```

**No general-purpose LLM is used.** Sections come from the notice's own headings and the answers are lines from the notice. Nothing is paraphrased, so an eligibility rule, an amount or a deadline can't be changed by a model. A missing section shows as "not stated in the document" instead of being guessed.

The pipeline lives in [`scheme_summariser.py`](scheme_summariser.py). This notebook calls it step by step.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import json
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Audio, Image, Markdown, display

import scheme_summariser as ss

load_dotenv()  # reads BODHAN_API_KEY (or the per-model keys) from .env
OUT = Path("outputs")
OUT.mkdir(exist_ok=True)
bodhan = ss.Bodhan(cache_dir=OUT / "cache")  # re-runs reuse translations and audio

## 1. A sample notice

The recipe ships a generator for a **synthetic** two-page notice: "Gramin Solar Pump Sahayata Yojana", a made-up scheme from the made-up state of Sampleland. It has a subsidy table, eligibility and exclusion rules, a document list, dates and a helpline, which is the usual shape of a real notification. Swap in any real PDF or a phone photo of a printed notice.

In [ ]:
!python make_sample_notice.py
NOTICE = Path("sample_data/sample_scheme_notice.pdf")
pages = ss.document_to_pages(NOTICE.read_bytes(), NOTICE.name)
print(len(pages), "pages")
display(Image(pages[0], width=520))

## 2. Read it with `indic-ocr`

`indic-ocr` takes one page image per request, so PDFs are rasterised at 150 DPI with PyMuPDF. Results are cached under `outputs/cache/` by image hash because OCR is billed per page. The model returns Markdown plus **layout blocks**, each with a type (`SectionHeader`, `Text`, `Table`, …) and a bounding box. The Markdown prints headings as plain lines, so `ocr_to_markdown()` rebuilds it from the blocks. Headings are marked, lines the page wrapped mid-sentence are re-joined, and a watermark repeated on every page is kept once.

In [ ]:
ocr = ss.ocr_pages(bodhan, pages)
for b in ocr[0]["blocks"][:10]:
    print(f"{b.get('order', ''):>2} {b.get('type', ''):14} {str(b.get('text', ''))[:60]!r}")

In [ ]:
markdown = ss.ocr_to_markdown(ocr)
print(markdown[:1200])

## 3. Build the scheme card, with no LLM

`build_card` splits the Markdown on headings (`#` lines, **bold** lines, `ALL CAPS` lines, lines ending in `:`, and short unpunctuated lines that name a section, such as "Eligibility") and maps each heading onto one of eight card fields by keyword: *About, Who can apply, Who cannot apply, What you get, Documents needed, How to apply, Important dates, Help and contact*. For a notice in Hindi or another Indic script, only the **headings** are translated to English for matching. The lines under them stay as printed.

Notices without clear headings still get dates and helplines, because lines carrying a date with "last/before/until", a phone number, an email or a URL are picked up directly.

In [ ]:
card = ss.build_card(markdown, bodhan)
display(Markdown(card.to_markdown()))
print(json.dumps(card.facts, ensure_ascii=False, indent=1))

## 4. Into the reader's language, plus a roman-script copy

Every line is translated with `indic-translate`. The labels and "not stated" message are translated too, so the whole card is in one language. Each **section is one request**, with its lines joined by newlines. The model keeps one output line per input line; the code checks this and falls back to line-by-line if it ever doesn't. That is about 10 calls per language instead of 40, which matters because each key allows about 120 requests a minute. The client also paces itself from the `x-ratelimit-*` headers.

The **roman-script** copy is how many people type on WhatsApp ("aapko kaun se documents chahiye"). It forwards well and reads on phones without Indic fonts. It comes straight from `indic-translate` with `script="roman"`, one step instead of translate-then-transliterate.

In [ ]:
cards = {lang: ss.localise_card(bodhan, card, lang) for lang in ["hi", "ta", "bn", "mr"]}
for lang, local in cards.items():
    (OUT / f"card_{lang}.md").write_text(local.to_markdown(), encoding="utf-8")
display(Markdown(cards["hi"].to_markdown()))

In [ ]:
print(cards["hi"].to_whatsapp()[:900])

In [ ]:
for lang in ["ta", "bn", "mr"]:
    local = cards[lang]
    print(f"{ss.LANGUAGES[lang][0]:8} | {local.labels['documents']}: {local.fields['documents'][0]}")

## 5. Read it aloud with `indic-speak`

Each section becomes its own clip, so a listener can jump straight to "Documents needed". `indic-speak` takes a sentence or two per request, so `speech_chunks` splits on sentence ends (`.`, `।`) and then on commas, and `join_wavs` stitches the 24 kHz PCM clips together with short pauses. The `Customer Care` style suits a helpdesk explaining a form.

In [ ]:
clips = ss.speak_card(bodhan, cards["hi"])
full = ss.join_wavs(clips.values(), gap_ms=600)
(OUT / "card_hi.wav").write_bytes(full)
print({k: f"{len(v) // 1024} KB" for k, v in clips.items()})
Audio(clips["documents"])

## 6. Ask a question by voice

To avoid shipping real voice recordings, the question audio is **synthesised** with `indic-speak` and then fed back through `indic-transcribe`, the same path a microphone recording takes in the app.

`ask()` transcribes the question, translates it to English, and routes it to a card field with question hints ("which documents" → *Documents needed*, "last date" → *Important dates*). It then ranks the card's lines by word overlap. The answer is those lines, translated back and spoken. Matching runs against the English view of the card, which is the card itself for an English notice. `english_card()` translates it once for a notice in another language.

In [ ]:
card_en = ss.english_card(bodhan, card)

question_audio = bodhan.speak("इस योजना के लिए मुझे कौन कौन से दस्तावेज़ चाहिए?", "hi", style=None)
reply = ss.ask(bodhan, card_en, "hi", audio=question_audio)

print("heard      :", reply.question)
print("in English :", reply.question_en)
print("section    :", reply.answer.field)
print("source     :", reply.answer.source_lines)
print("answer     :", reply.text)
Audio(reply.audio)

Typed questions work too, including roman script typed on an English keyboard: `indic-transliterate` (`script="native"`) converts it to native script first. A question the notice doesn't cover gets an honest "the notice does not answer this" instead of a guess.

In [ ]:
for question in ["kya kirayedar kisan bhi apply kar sakta hai?", "aakhri tareekh kya hai?", "kya isme loan bhi milta hai?"]:
    r = ss.ask(bodhan, card_en, "hi", text=question, speak=False)
    print(f"{question}\n  -> [{r.answer.field or 'not found'}] {r.text}\n")

## 7. Run it as an app or a CLI

```bash
streamlit run app.py                                                           # upload, pick a language, listen, ask by mic
python scheme_summariser.py sample_data/sample_scheme_notice.pdf --lang hi ta  # writes outputs/card_*.md, *_whatsapp.txt, *.wav
```

**Where to take it next:** a WhatsApp bot with the same `ask()` behind it, an IVR line built on [Twilio/Exotel](../../integrations/README.md), or a CSC-operator view that prints the card for walk-in citizens.